# What does a per-segment polynomial detrend remove from the low-frequency spectrum?

A polynomial trend is not noise, but a windowed segment cannot tell the difference. On a single segment a slow trend looks like an offset, a slope, and a curvature. The window turns each of those into a smooth spectrum piled into the lowest bins, often far above the noise floor. Subtracting a polynomial from the segment before the transform removes exactly that power, provided the polynomial covers what the trend looks like on a segment of that length.

SpecKit fits the polynomial independently on every segment, and `order` chooses it. `order=-1` applies the window and subtracts nothing. `order=0` subtracts the mean. At `order=1` and `order=2` the segment is projected onto an orthonormal basis $Q$, built by `_build_Q` on $t \in [-1, 1]$, and

$$y_{\rm detr} = y - Q Q^{\mathsf T} y$$

removes a straight line, or a line and a quadratic. A trend that is quadratic over the whole record is quadratic on every piece of it, so `order=2` removes it completely. An order that is one too low leaves the next power of $t$ to leak into the first bins.

Logarithmic scheduling puts the longest segments at the lowest frequencies, which is where a quadratic has the most time to grow and where the leaked power lands in the bins one most wants to trust.

The series is 16 s at $f_s = 1024\,\mathrm{Hz}$: sinusoids of amplitude 0.2 at 7.5 Hz and 0.1 at 55 Hz, white noise of standard deviation 0.003, and the trend $0.8 - 0.5\,t + 0.01\,t^2$ with $t$ in seconds. `compute_spectrum(x, fs, order=o)` is run at all four orders. The rest is the default: a Kaiser window at 200 dB peak sidelobes, its recommended overlap, and `vectorized_ltf`. The spectra are one-sided ASDs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from speckit import compute_spectrum, default_rc

plt.rcParams.update(default_rc)

In [ ]:
# Generate synthetic data with quadratic trend
np.random.seed(7)
fs = 1024.0  # Hz
T = 16.0  # seconds
N = int(T * fs)
t = np.arange(N) / fs

# A couple of tones + white noise
noise = 0.003 * np.random.randn(N)
sig = 0.2 * np.sin(2 * np.pi * 7.5 * t) + 0.1 * np.sin(2 * np.pi * 55.0 * t)

# Quadratic trend a0 + a1*t + a2*t^2
a0, a1, a2 = 0.8, -0.5, 0.01
trend = a0 + a1 * t + a2 * (t**2)

x = sig + noise + trend

# Compute spectra for orders -1, 0, 1, 2
orders = [-1, 0, 1, 2]
results = {}
for o in orders:
    res = compute_spectrum(x, fs, order=o)
    results[o] = (res.f, res.asd)

# %% [4] Plot ASD overlays
plt.figure(figsize=(7.5, 5))
for o in orders:
    f, A = results[o]
    if o == 2:
        ls = "--"
    else:
        ls = "-"
    plt.loglog(f, A, label=f"order={o}", lw=2, ls=ls)
plt.xlabel("Frequency [Hz]")
plt.ylabel("ASD [units/√Hz]")
plt.title("Effect of polynomial detrending on ASD")
plt.grid(True, which="both")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from speckit import compute_single_bin

def direct_coefficient(segment, window, omega, order):
    """Direct sum after the documented detrend. Does not call Goertzel."""
    y = np.array(segment, dtype=np.float64, copy=True)
    if order == 0:
        y = y - y.mean()
    elif order in (1, 2):
        t_seg = np.linspace(-1.0, 1.0, y.size)
        if order == 1:
            basis = np.column_stack([np.ones(y.size), t_seg])
        else:
            basis = np.column_stack([np.ones(y.size), t_seg, t_seg * t_seg])
        q, _ = np.linalg.qr(basis, mode="reduced")
        y = y - q @ (q.T @ y)
    n = np.arange(y.size, dtype=np.float64)
    return np.dot(y * window, np.exp(-1j * omega * n))

freq = 40.0
length = 256
window = np.hanning(length)
omega = 2.0 * np.pi * freq / fs
for order in (-1, 0, 1, 2):
    single = compute_single_bin(
        x, fs, freq=freq, L=length, order=order, win="hann", olap=0.0
    )
    starts = np.asarray(single.D[0], dtype=np.int64)
    coefficients = [
        direct_coefficient(x[s : s + length], window, omega, order) for s in starts
    ]
    direct_power = np.mean(np.abs(coefficients) ** 2)
    relative = abs(float(single.XX[0]) - direct_power) / direct_power
    print(f"order {order}: relative |XX - mean(|X_direct|^2)| = {relative:.3e}")


## Reading the result

The figure overlays the four ASDs, with `order=2` dashed. At the two tones and along the high-frequency floor, all four should coincide. The trend has no content there, and detrending a segment should not move a tone. At the bottom of the band they should separate in order. `order=-1` carries the offset, the ramp, and the curvature. `order=0` removes the offset. `order=1` removes the ramp. `order=2` removes the curvature as well.

The reference for `order=2` is the injected noise, not the other curves. White noise of standard deviation $\sigma$ has the one-sided ASD $\sigma\sqrt{2/f_s} = 0.003\sqrt{2/1024} \approx 1.3\times 10^{-4}$ units$/\sqrt{\mathrm{Hz}}$. `order=2` should reach that floor down to the lowest bins. The lower orders should rise above it there, and rejoin it once the leaked trend has dropped below the noise.

The record is 16 s long, so the first bin is near $f_s/N = 1/16\,\mathrm{Hz}$. The height of a tone in this plot is an ASD in a bin whose width changes with frequency. It is not the amplitude of the tone.

The last cell leaves the Kaiser schedule and checks one interior bin, 40 Hz with a Hann window of 256 samples and no overlap, at each `order`. The reference is a direct sum `sum y[n] exp(-i omega n)` after the same detrend and window. It does not call the Goertzel kernel. The production `XX` is the mean of $|X|^2$, so the common Goertzel phase `exp(i omega (L-1))` cancels. The relative difference should sit at roundoff for every order. Fractional bins, a near-zero frequency, and a long segment are in `notebooks/1.5_direct-dft.ipynb`.